# Similitud todos-contra-todos · `intfloat/multilingual-e5-base`

Compara las **378 filas** de `data/original/*.csv` entre sí con embeddings semánticos.

Texto por fila (igual que producción / `buildInstructionText`):

```text
Concejalía: {case_group}. Subtema: {case_subgroup}. {instruction}
```

E5 document↔documento usa el prefijo `passage: `.

**Nota sobre umbrales:** con este corpus la similitud coseno de E5 se concentra alta (media ≈ 0.86).
Un umbral de 0.85 es casi la mediana y no sirve. Usa **≥ 0.97** para casi-duplicados y **≥ 0.95** para candidatos a revisar.

## 1. Dependencias

```bash
pip install -r data/notebooks/requirements.txt
```

La primera carga de `intfloat/multilingual-e5-base` descarga ~1 GB a la caché de Hugging Face (`~/.cache/huggingface`).

In [ ]:
from pathlib import Path
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

_cwd = Path.cwd().resolve()
ROOT = None
for candidate in (_cwd, _cwd.parent, _cwd.parent.parent, *_cwd.parents):
    if (candidate / "data" / "original").is_dir():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("No se encontró data/original desde el cwd del kernel")

DATA_DIR = ROOT / "data" / "original"
OUT_DIR = ROOT / "data" / "notebooks" / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "intfloat/multilingual-e5-base"
BATCH_SIZE = 32
# Calibrado en este corpus (E5): media≈0.86, p95≈0.94, p99≈0.96
SIM_THRESHOLD = 0.97

print(f"ROOT     = {ROOT}")
print(f"DATA_DIR = {DATA_DIR}")
print(f"OUT_DIR  = {OUT_DIR}")
assert DATA_DIR.is_dir(), f"No existe {DATA_DIR}"

## 2. Cargar CSV y construir el texto a embeber

In [ ]:
def build_instruction_text(row: pd.Series) -> str:
    """Misma construcción que src/services/voyageEmbeddings.js → buildInstructionText."""
    return (
        f"Concejalía: {row['case_group']}. "
        f"Subtema: {row['case_subgroup']}. "
        f"{row['instruction']}"
    )


frames = []
for csv_path in sorted(DATA_DIR.glob("*.csv")):
    part = pd.read_csv(csv_path, dtype=str).fillna("")
    part["source_file"] = csv_path.name
    part["row_in_file"] = np.arange(len(part))
    frames.append(part)

df = pd.concat(frames, ignore_index=True)
df["text"] = df.apply(build_instruction_text, axis=1)
df["text_e5"] = "passage: " + df["text"]

print(f"Filas cargadas: {len(df)}")
print(df.groupby("source_file").size().rename("n").to_string())
df[["source_file", "case_group", "case_subgroup"]].head(3)

## 3. Cargar modelo y generar embeddings

In [ ]:
model = SentenceTransformer(MODEL_NAME)
print(f"Modelo: {MODEL_NAME}")
print(f"Dispositivo: {model.device}")

embeddings = model.encode(
    df["text_e5"].tolist(),
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
)
embeddings = np.asarray(embeddings, dtype=np.float32)
print(f"Shape embeddings: {embeddings.shape}")

## 4. Matriz de similitud todos-contra-todos

In [ ]:
sim = embeddings @ embeddings.T
np.fill_diagonal(sim, 0.0)

off = sim[np.triu_indices_from(sim, k=1)]
print(f"Matriz: {sim.shape}  |  pares únicos: {len(off):,}")
print(
    f"media={off.mean():.4f}  p50={np.percentile(off, 50):.4f}  "
    f"p95={np.percentile(off, 95):.4f}  p99={np.percentile(off, 99):.4f}  máx={off.max():.4f}"
)

## 5. Pares ≥ umbral (casi-duplicados)

In [ ]:
ii, jj = np.triu_indices_from(sim, k=1)
scores = sim[ii, jj]
mask = scores >= SIM_THRESHOLD

pairs = (
    pd.DataFrame({"i": ii[mask], "j": jj[mask], "similarity": scores[mask]})
    .sort_values("similarity", ascending=False)
    .reset_index(drop=True)
)

meta_cols = ["source_file", "case_group", "case_subgroup", "instruction"]
left = df.loc[pairs["i"], meta_cols].reset_index(drop=True).add_prefix("a_")
right = df.loc[pairs["j"], meta_cols].reset_index(drop=True).add_prefix("b_")
pairs_full = pd.concat([pairs, left, right], axis=1)

out_csv = OUT_DIR / f"pares_similares_e5_ge_{SIM_THRESHOLD:.2f}.csv"
pairs_full.to_csv(out_csv, index=False, encoding="utf-8-sig")

print(f"Pares ≥ {SIM_THRESHOLD}: {len(pairs_full)}")
print(f"Guardado: {out_csv}")
pairs_full.head(25)

## 6. Top-N pares globales

In [ ]:
TOP_N = 40

order = np.argsort(scores)[::-1][:TOP_N]
top = pd.DataFrame({"i": ii[order], "j": jj[order], "similarity": scores[order]})
left = df.loc[top["i"], meta_cols].reset_index(drop=True).add_prefix("a_")
right = df.loc[top["j"], meta_cols].reset_index(drop=True).add_prefix("b_")
top_full = pd.concat([top, left, right], axis=1)

out_top = OUT_DIR / f"top_{TOP_N}_pares_e5.csv"
top_full.to_csv(out_top, index=False, encoding="utf-8-sig")
print(f"Guardado: {out_top}")
top_full

## 7. Inspeccionar un par

Cambia `PAIR_RANK` (0 = el más similar del top).

In [ ]:
PAIR_RANK = 0

row = top_full.iloc[PAIR_RANK]
print(f"similarity = {row['similarity']:.4f}")
print("\n--- A ---")
print(f"[{row['a_source_file']}] {row['a_case_group']} › {row['a_case_subgroup']}")
print(row["a_instruction"][:900])
print("\n--- B ---")
print(f"[{row['b_source_file']}] {row['b_case_group']} › {row['b_case_subgroup']}")
print(row["b_instruction"][:900])

## 8. Heatmap: similitud media entre archivos CSV

In [ ]:
files = sorted(df["source_file"].unique())
file_to_idx = {f: np.where(df["source_file"].values == f)[0] for f in files}

cross = pd.DataFrame(index=files, columns=files, dtype=float)
for fa, fb in itertools.product(files, repeat=2):
    ia, ib = file_to_idx[fa], file_to_idx[fb]
    block = sim[np.ix_(ia, ib)]
    if fa == fb:
        if len(ia) < 2:
            cross.loc[fa, fb] = np.nan
        else:
            m = ~np.eye(len(ia), dtype=bool)
            cross.loc[fa, fb] = block[m].mean()
    else:
        cross.loc[fa, fb] = block.mean()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cross.astype(float).values, vmin=0.7, vmax=0.95, cmap="viridis")
ax.set_xticks(range(len(files)))
ax.set_yticks(range(len(files)))
short = [f.replace(".csv", "") for f in files]
ax.set_xticklabels(short, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(short, fontsize=8)
ax.set_title("Similitud media entre archivos (E5-base)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()

heatmap_path = OUT_DIR / "heatmap_similitud_media_por_archivo.png"
fig.savefig(heatmap_path, dpi=140)
print(f"Guardado: {heatmap_path}")
plt.show()
cross.round(3)

## 9. Guardar embeddings (reanálisis sin re-embeber)

In [ ]:
emb_path = OUT_DIR / "embeddings_e5_base.npy"
meta_path = OUT_DIR / "embeddings_meta.csv"

np.save(emb_path, embeddings)
df[["source_file", "row_in_file", "case_group", "case_subgroup", "text"]].to_csv(
    meta_path, index=True, index_label="row_id", encoding="utf-8-sig"
)

print(f"Embeddings: {emb_path} ({emb_path.stat().st_size / 1e6:.1f} MB)")
print(f"Meta:       {meta_path}")

## Notas

| Umbral | Uso en este corpus |
|--------|--------------------|
| 0.99 | Casi idénticos (p. ej. mismo trámite, año distinto) |
| 0.97 | Casi-duplicados / plantillas muy cercanas |
| 0.95 | Candidatos a revisar |
| 0.85 | **No usar** (≈ mediana) |

Producción sigue con Voyage (`voyage-4-lite`); este notebook es análisis offline.
Salidas en `data/notebooks/output/` (ignorado por git).